# V × T Delay Surface Viewer (2-D V×T dataset)

Interactive notebook for inspecting per-cell test data from `gnn_dataset_2d`.
Loads a cell `.pth` file (format `unified_4d_VT`) and lets you plot V-T-Delay
surfaces (heatmap and 3-D), single-axis slices, distribution stats, and
per-(V,T) heatmaps.

Built for the TAMEL-style commercial-28nm setup:
- 61 voltage points (0.60 V → 1.20 V, step 0.01 V)
- 5 test temperatures (0, 25, 50, 75, 100 °C)
- Each cell has many tasks = (timing arc, slew, load) combinations.

**Reference paper:** `claude_context/TAMEL.pdf`.

**Tip for 3-D plots:** the cells use `%matplotlib notebook` (or `%matplotlib widget`
if `ipympl` is installed) inside the 3-D sections so you can rotate the surface
by dragging. If neither backend is available, the static `inline` 3-D plot still
works — just no rotation.

## 1. Setup

In [ ]:
import os, glob
import numpy as np
import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 — registers 3-D projection
%matplotlib inline
plt.rcParams['figure.dpi'] = 110

REPO_ROOT = '/home/tkdgn2907/Deepsets_test/MAML'
TEST_DIR = f'{REPO_ROOT}/Projects/dataset_all/gnn_dataset_2d/test_by_cell_stage_aware_2d'
TRAIN_PATH = f'{REPO_ROOT}/Projects/dataset_all/gnn_dataset_2d/train_cell_stage_aware_2d.pth'

# All available test cells
ALL_CELLS = sorted(os.path.basename(p).replace('.pth', '')
                   for p in glob.glob(f'{TEST_DIR}/*.pth'))
print(f'{len(ALL_CELLS)} test cells available:')
for c in ALL_CELLS[:20]:
    print('  ', c)
if len(ALL_CELLS) > 20:
    print(f'  ... and {len(ALL_CELLS)-20} more')

## 2. Load a cell

Change `CELL_NAME` to the cell you want to inspect.

In [ ]:
CELL_NAME = 'SDFSNQD0BWP30P140'   # try AN4D0BWP30P140 (combinational) for comparison

path = f'{TEST_DIR}/{CELL_NAME}.pth'
data = torch.load(path, weights_only=False, map_location='cpu')
outputs    = data['outputs']           # [V, T, num_tasks]
voltages   = data['voltages'].numpy()
temperatures = data['temperatures'].numpy()
delay_types = data.get('delay_types', None)
output_names = data.get('output_names', None)
task_corners = data.get('task_corners', None)

V, T, N = outputs.shape
print(f'cell             : {CELL_NAME}')
print(f'shape            : {tuple(outputs.shape)}  (V × T × num_tasks)')
print(f'voltages [V]     : {voltages[0]:.2f} → {voltages[-1]:.2f}  ({V} points)')
print(f'temperatures [°C]: {temperatures.tolist()}')
print(f'num tasks        : {N}')
if task_corners is not None:
    print(f'corners present  : {sorted(set(task_corners))}')
if delay_types is not None:
    print(f'delay types      : {sorted(set(delay_types))}')

## 3. Single-task V × T heatmap (2-D view)

Color = delay magnitude. Black regions usually mean `output == 0` —
common in sequential cells where setup time is violated at low V.

In [ ]:
TASK_IDX = 0   # any integer in [0, N); try N//2 or random

def plot_vt_heatmap(task_idx, ax=None, cmap='viridis', title=None):
    s = outputs[:, :, task_idx].numpy()
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(6.5, 4))
    im = ax.imshow(s, aspect='auto', origin='lower',
                   extent=[temperatures[0], temperatures[-1],
                           voltages[0], voltages[-1]],
                   cmap=cmap)
    plt.colorbar(im, ax=ax, label='Delay (normalized)')
    ax.set_xlabel('Temperature [°C]')
    ax.set_ylabel('Voltage [V]')
    info = []
    if title is not None:
        info.append(title)
    info.append(f'{CELL_NAME} — task {task_idx}')
    if delay_types is not None and task_corners is not None:
        info.append(f'corner={task_corners[task_idx]} delay_type={delay_types[task_idx]}')
    info.append(f'range=[{s.min():.3e}, {s.max():.3e}]  median={np.median(s):.3e}')
    ax.set_title('\n'.join(info), fontsize=9)
    return ax

plot_vt_heatmap(TASK_IDX);
plt.show()

## 4. Single-task 3-D surface view

Same data as section 3, drawn as a 3-D surface. Z-axis = delay magnitude.

Run `%matplotlib notebook` (cell below) once to enable mouse-drag rotation.
If your Jupyter doesn't support that backend, you can also try
`%matplotlib widget` (requires `pip install ipympl`). Static `inline` is fine
for screenshots but is not rotatable.

In [ ]:
# Try rotatable backend — fall back to inline if not available.
try:
    get_ipython().run_line_magic('matplotlib', 'notebook')
    print('Backend = notebook (drag to rotate)')
except Exception:
    try:
        get_ipython().run_line_magic('matplotlib', 'widget')
        print('Backend = widget (drag to rotate, requires ipympl)')
    except Exception:
        print('Falling back to inline — 3-D plot will be static')
        get_ipython().run_line_magic('matplotlib', 'inline')

In [ ]:
TASK_IDX = 0

def plot_vt_surface_3d(task_idx, ax=None, cmap='viridis',
                       elev=25, azim=-50, title=None,
                       show_wire=False):
    s = outputs[:, :, task_idx].numpy()
    # Build mesh — note T is on X axis, V is on Y axis (matches heatmap)
    Tg, Vg = np.meshgrid(temperatures, voltages)
    if ax is None:
        fig = plt.figure(figsize=(8, 6))
        ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(Tg, Vg, s, cmap=cmap,
                            linewidth=0, antialiased=True,
                            rcount=V, ccount=T, alpha=0.92)
    if show_wire:
        ax.plot_wireframe(Tg, Vg, s, color='k', linewidth=0.3, alpha=0.4,
                          rcount=12, ccount=T)
    ax.set_xlabel('Temperature [°C]')
    ax.set_ylabel('Voltage [V]')
    ax.set_zlabel('Delay')
    ax.view_init(elev=elev, azim=azim)
    info = []
    if title is not None:
        info.append(title)
    info.append(f'{CELL_NAME} — task {task_idx}')
    if delay_types is not None and task_corners is not None:
        info.append(f'corner={task_corners[task_idx]} delay_type={delay_types[task_idx]}')
    info.append(f'range=[{s.min():.3e}, {s.max():.3e}]  median={np.median(s):.3e}')
    ax.set_title('\n'.join(info), fontsize=9)
    fig = ax.figure
    cbar = fig.colorbar(surf, ax=ax, shrink=0.65, label='Delay')
    return ax

plot_vt_surface_3d(TASK_IDX, show_wire=True);
plt.show()

Toggle the wire mesh, change viewing angle, or pick a different task:

In [ ]:
# Examples — feel free to change.
plot_vt_surface_3d(0,        elev=20, azim=-60, show_wire=False);
plt.show()

plot_vt_surface_3d(N // 2,   elev=40, azim=-120, show_wire=True);
plt.show()

## 5. Grid of random tasks (heatmap)

Surface variability across tasks for the same cell. Heatmap is faster to
render than 3-D when showing many panels.

In [ ]:
# Switch back to inline for the multi-panel heatmap grid (faster, screenshotable)
get_ipython().run_line_magic('matplotlib', 'inline')

N_SAMPLES = 8
rng = np.random.default_rng(0)
sample_tasks = rng.choice(N, size=N_SAMPLES, replace=False)

ncols = 4
nrows = (N_SAMPLES + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*4.5, nrows*3.5))
axes = np.atleast_2d(axes).ravel()
for ax, ti in zip(axes, sample_tasks):
    plot_vt_heatmap(int(ti), ax=ax)
for ax in axes[N_SAMPLES:]:
    ax.set_visible(False)
fig.suptitle(f'V × T delay surfaces — {N_SAMPLES} random tasks of {CELL_NAME}',
             fontsize=12)
fig.tight_layout()
plt.show()

## 6. Grid of random tasks (3-D)

Slower than section 5 but shows shape more vividly. Reduce `N_SAMPLES` if it
lags.

In [ ]:
N_SAMPLES_3D = 4
rng = np.random.default_rng(0)
sample_tasks_3d = rng.choice(N, size=N_SAMPLES_3D, replace=False)

ncols = N_SAMPLES_3D
fig = plt.figure(figsize=(ncols * 5.0, 4.5))
for col, ti in enumerate(sample_tasks_3d):
    ax = fig.add_subplot(1, ncols, col + 1, projection='3d')
    plot_vt_surface_3d(int(ti), ax=ax, elev=25, azim=-55, show_wire=True)
fig.suptitle(f'V × T delay 3-D surfaces — {N_SAMPLES_3D} random tasks of {CELL_NAME}',
             fontsize=12)
fig.tight_layout(); plt.show()

## 7. Single-axis slices

Fixed T → delay vs V (one curve per T), and fixed V → delay vs T.

In [ ]:
TASK_IDX = 0

s = outputs[:, :, TASK_IDX].numpy()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

for t_idx, t_val in enumerate(temperatures):
    ax1.plot(voltages, s[:, t_idx], '-o', markersize=2.5,
             label=f'T={t_val:g}°C')
ax1.set_xlabel('Voltage [V]'); ax1.set_ylabel('Delay')
ax1.set_title(f'{CELL_NAME} — task {TASK_IDX}: delay vs V (one line per T)')
ax1.legend(fontsize=8, ncol=2)
ax1.grid(True, alpha=0.3)

for v_idx in [0, 5, 15, 30, 45, 55, 60]:
    ax2.plot(temperatures, s[v_idx, :], '-o', markersize=4,
             label=f'V={voltages[v_idx]:.2f}V')
ax2.set_xlabel('Temperature [°C]'); ax2.set_ylabel('Delay')
ax2.set_title(f'{CELL_NAME} — task {TASK_IDX}: delay vs T (one line per V)')
ax2.legend(fontsize=8, ncol=2)
ax2.grid(True, alpha=0.3)

fig.tight_layout(); plt.show()

## 8. Surface smoothness diagnostics

Aggregate over all tasks of this cell:
- **mono(dV ≤ 0)**: fraction of (V, T) pairs where delay decreases with V (normal CMOS).
- **mono(dT ≥ 0)**: fraction of (V, T) pairs where delay increases with T (normal at high V; broken under ITD at low V).
- **zero fraction**: fraction of (V, T, task) entries that are exactly zero (sequential cells with timing violations).

In [ ]:
out_np = outputs.numpy()
dV = np.diff(out_np, axis=0)
dT = np.diff(out_np, axis=1)
mono_V_overall = float((dV <= 0).mean())
mono_T_overall = float((dT >= 0).mean())
zero_frac      = float((out_np == 0).mean())
task_ranges    = out_np.max(axis=(0, 1)) - out_np.min(axis=(0, 1))
task_medians   = np.median(out_np, axis=(0, 1))

print(f'{CELL_NAME} — {N} tasks')
print(f'  mono(dV ≤ 0)   : {mono_V_overall:6.2%}   (1.0 = perfectly monotonic-decreasing in V)')
print(f'  mono(dT ≥ 0)   : {mono_T_overall:6.2%}   (1.0 = no ITD, monotonic-increasing in T)')
print(f'  zero fraction  : {zero_frac:6.2%}   (sequential cells often have nonzero zero-fraction)')
print(f'  per-task range : median={np.median(task_ranges):.3e}, [{task_ranges.min():.3e}, {task_ranges.max():.3e}]')
print(f'  per-task median: median={np.median(task_medians):.3e}')

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].hist(task_ranges, bins=40, color='C0', alpha=0.85)
axes[0].set_xlabel('per-task (max − min) delay')
axes[0].set_ylabel('# tasks')
axes[0].set_title(f'{CELL_NAME}: distribution of per-task delay range')
axes[0].grid(True, alpha=0.3)
axes[1].hist(task_medians, bins=40, color='C1', alpha=0.85)
axes[1].set_xlabel('per-task median delay')
axes[1].set_ylabel('# tasks')
axes[1].set_title(f'{CELL_NAME}: distribution of per-task median')
axes[1].grid(True, alpha=0.3)
fig.tight_layout(); plt.show()

## 9. Compare 4 cells — heatmap and 3-D side-by-side

Useful for combinational vs sequential comparison (AN4D0, OA21D0 vs
SDFSNQD0, DFCNQD1). Top row = heatmap, bottom row = 3-D.

In [ ]:
CELLS_TO_COMPARE = ['AN4D0BWP30P140', 'OA21D0BWP30P140',
                    'SDFSNQD0BWP30P140', 'DFCNQD1BWP30P140']
TASK_IDX_CMP = 0

SEQ_PFX = ('SDF', 'DFC', 'DFF', 'LDF', 'LSC', 'DFR', 'DFS')
fig = plt.figure(figsize=(5.0 * len(CELLS_TO_COMPARE), 9))
for col, cn in enumerate(CELLS_TO_COMPARE):
    p = f'{TEST_DIR}/{cn}.pth'
    d2 = torch.load(p, weights_only=False, map_location='cpu')
    out2 = d2['outputs']
    v2 = d2['voltages'].numpy()
    t2 = d2['temperatures'].numpy()
    n2 = out2.shape[2]
    ti = TASK_IDX_CMP if TASK_IDX_CMP < n2 else 0
    s = out2[:, :, ti].numpy()
    is_seq = any(cn.upper().startswith(p) for p in SEQ_PFX)
    cmap = 'magma' if is_seq else 'viridis'

    # Top row: heatmap
    ax_h = fig.add_subplot(2, len(CELLS_TO_COMPARE), col + 1)
    im = ax_h.imshow(s, aspect='auto', origin='lower',
                     extent=[t2[0], t2[-1], v2[0], v2[-1]], cmap=cmap)
    fig.colorbar(im, ax=ax_h, label='Delay', shrink=0.8)
    ax_h.set_xlabel('T [°C]'); ax_h.set_ylabel('V [V]')
    ax_h.set_title(f'{"SEQ" if is_seq else "COMB"} — {cn}\ntask {ti} (heatmap)',
                    fontsize=9)

    # Bottom row: 3-D surface
    ax_s = fig.add_subplot(2, len(CELLS_TO_COMPARE),
                            len(CELLS_TO_COMPARE) + col + 1,
                            projection='3d')
    T2g, V2g = np.meshgrid(t2, v2)
    surf = ax_s.plot_surface(T2g, V2g, s, cmap=cmap,
                              linewidth=0, antialiased=True, alpha=0.92,
                              rcount=v2.size, ccount=t2.size)
    fig.colorbar(surf, ax=ax_s, label='Delay', shrink=0.6)
    ax_s.set_xlabel('T [°C]'); ax_s.set_ylabel('V [V]'); ax_s.set_zlabel('Delay')
    ax_s.view_init(elev=25, azim=-55)
    ax_s.set_title(f'{cn} — task {ti} (3-D)', fontsize=9)

fig.suptitle('V × T delay surfaces — combinational vs sequential',
             fontsize=12)
fig.tight_layout(); plt.show()

## 10. Train-set composition (combinational vs sequential)

Quick sanity: are sequential cells present in the train set at all?

In [ ]:
train = torch.load(TRAIN_PATH, weights_only=False, map_location='cpu', mmap=True)
tnames = train['cell_names']
uniq = sorted(set(tnames))

SEQ_PREFIXES = ('SDF', 'DFC', 'DFF', 'DFR', 'DFS', 'LDF', 'LSC', 'SDFC', 'SDFR', 'SDFS')
def is_seq(name):
    return any(name.upper().startswith(p) for p in SEQ_PREFIXES)

n_total = len(tnames)
n_seq_tasks = sum(1 for n in tnames if is_seq(n))
n_seq_cells = sum(1 for n in uniq if is_seq(n))

print(f'Train set: {n_total} tasks across {len(uniq)} unique cells')
print(f'  Sequential tasks : {n_seq_tasks} ({100*n_seq_tasks/n_total:.2f}%)')
print(f'  Sequential cells : {n_seq_cells} / {len(uniq)}')
if n_seq_cells:
    for c in uniq:
        if is_seq(c):
            print('    ', c)

## 11. Per-(V, T) median across tasks — cell fingerprint (heatmap + 3-D)

Bird's-eye view: median delay over all tasks of this cell at each (V, T).
Captures the "average shape" of the cell's V×T behavior, independent of
task slew/load. Often the cleanest signature for cell-vs-cell comparison.

In [ ]:
median_surface = np.median(outputs.numpy(), axis=2)   # [V, T]

fig = plt.figure(figsize=(13.5, 4.5))

ax_h = fig.add_subplot(1, 2, 1)
im = ax_h.imshow(median_surface, aspect='auto', origin='lower',
                 extent=[temperatures[0], temperatures[-1],
                         voltages[0], voltages[-1]],
                 cmap='viridis')
fig.colorbar(im, ax=ax_h, label='Median delay')
ax_h.set_xlabel('Temperature [°C]'); ax_h.set_ylabel('Voltage [V]')
ax_h.set_title(f'{CELL_NAME}: median delay (heatmap)')

ax_s = fig.add_subplot(1, 2, 2, projection='3d')
Tg, Vg = np.meshgrid(temperatures, voltages)
surf = ax_s.plot_surface(Tg, Vg, median_surface, cmap='viridis',
                          linewidth=0, antialiased=True, alpha=0.92,
                          rcount=V, ccount=T)
fig.colorbar(surf, ax=ax_s, label='Median delay', shrink=0.7)
ax_s.set_xlabel('T [°C]'); ax_s.set_ylabel('V [V]'); ax_s.set_zlabel('Delay')
ax_s.view_init(elev=25, azim=-55)
ax_s.set_title(f'{CELL_NAME}: median delay (3-D)')

fig.tight_layout(); plt.show()

---

End of notebook. Useful next steps:
- Set `CELL_NAME` to any cell from `ALL_CELLS` and re-run from section 2.
- Tweak `elev` / `azim` in any 3-D cell to find the best viewing angle.
- For genuinely interactive 3-D in Jupyter Lab, install `ipympl` and use
  `%matplotlib widget` instead of `%matplotlib notebook`.
- Once validation is rerun with `--save_results`, you can load the prediction
  `.npy` files (e.g., `data_result_npy_directory_final/*_pred.npy`) and plot the
  predicted surface alongside the actual one.